# Hindsight Memory Types — a hands-on walkthrough

This notebook talks to Hindsight's REST API directly — no LLM, no agent, no chat UI.
That is the point: it removes every source of randomness so each step always produces
the same result, and you can watch exactly what Hindsight does with your data.

It answers one question: **when you call `retain`, what actually gets created, and how
does it turn into the richer things you see in the [Admin UI](http://localhost:9999)?**

Run the cells top to bottom. After each one, open the Admin UI, pick the
`explainer-demo` bank, and look at what changed.

For the *agent decides when to use these tools* side of the story (an LLM deciding to
call `retain`/`recall`/`reflect` mid-conversation), see the main chat app one level up —
this notebook deliberately skips that part.

In [ ]:
import requests
import time
import json

BASE_URL = "http://localhost:8888"
BANK_ID = "explainer-demo"


def get(path, **params):
    r = requests.get(f"{BASE_URL}{path}", params=params)
    r.raise_for_status()
    return r.json()


def post(path, body=None):
    r = requests.post(f"{BASE_URL}{path}", json=body or {})
    r.raise_for_status()
    return r.json()


def delete(path):
    r = requests.delete(f"{BASE_URL}{path}")
    r.raise_for_status()


def show_memories(fact_type=None):
    data = get(f"/v1/default/banks/{BANK_ID}/memories/list")
    items = data.get("items", [])
    if fact_type:
        items = [m for m in items if m.get("fact_type") == fact_type]
    print(f"{len(items)} kayit:")
    for m in items:
        print(f"  [{m.get('fact_type'):<11}] {m.get('text')}")
    return items


def wait_until(check_fn, timeout=90, interval=3, label="sonuc"):
    """Poll check_fn() every `interval` seconds until it returns a truthy value,
    instead of guessing a fixed sleep duration — background processing time in
    Hindsight varies (consolidation, mental-model generation, etc)."""
    waited = 0
    while waited < timeout:
        result = check_fn()
        if result:
            print(f"Hazir ({waited}s sonra).")
            return result
        print(f"  ...{label} icin bekleniyor ({waited}s)")
        time.sleep(interval)
        waited += interval
    print(f"UYARI: {timeout}s icinde {label} hazir olmadi. Admin UI'dan elle kontrol et.")
    return None


# Health check — make sure Hindsight is actually running before we start.
health = requests.get(f"{BASE_URL}/health", timeout=3).json()
print("Hindsight health:", health)

## 0. Clean slate

We use our own bank (`explainer-demo`), separate from the chat app's `destek-hatti-demo`
bank, so this notebook never mixes data with a live demo. Delete anything left over from
a previous run so the counts below are easy to follow.

In [ ]:
try:
    delete(f"/v1/default/banks/{BANK_ID}/memories")
    print("Onceki kayitlar silindi.")
except requests.HTTPError as e:
    print("Silinecek bir sey yoktu (ya da hata):", e)

# Mental model'i de temizle — "memories" endpoint'i onu silmiyor, ve ayni id ile
# tekrar olusturmaya calismak 500 hatasi veriyor.
try:
    delete(f"/v1/default/banks/{BANK_ID}/mental-models/musteri-profili")
    print("Onceki mental model silindi.")
except requests.HTTPError:
    pass  # yoktu, sorun degil

show_memories()

## 1. `retain` → World Facts

`retain` is the simplest operation: "remember this." Under the hood, Hindsight runs the
content through its own LLM to pull out one or more atomic, structured facts. Those land
in the **World Facts** tab — "objective facts about the world received from external
sources."

We are not calling a `retain` *tool* through an agent here — we are calling the same REST
endpoint the MCP tool calls internally, directly.

In [ ]:
result = post(f"/v1/default/banks/{BANK_ID}/memories", {
    "items": [
        {
            "content": "Ahmet, telefon yerine e-posta ile iletisim kurulmasini tercih ediyor.",
            "context": "iletisim tercihi",
        }
    ]
})
print(json.dumps(result, indent=2, ensure_ascii=False))

In [ ]:
# Bu tek retain cagrisindan World Facts'te ne olustu?
show_memories(fact_type="world")

Open the Admin UI now and look at **World Facts** — you should see the same fact there,
with an `Ahmet` entity tag already attached. Hindsight extracted the entity for you.

## 2. More facts, then `consolidate` → Observations

A single fact does not usually produce an **Observation** — observations are Hindsight's
own synthesis, "patterns, preferences, and learnings that emerge from *accumulated*
evidence." We need more than one fact for a pattern to emerge, and we need to trigger a
consolidation pass (this normally also happens on a schedule/automatically; we trigger it
by hand here so the timing is predictable in a live walkthrough).

In [ ]:
more_facts = [
    {"content": "Ahmet, gecen ay yasadigi kargo gecikmesi sikayeti icin ozur e-postasi aldi.",
     "context": "kargo sikayeti"},
    {"content": "Ahmet'in premium musteri hesabi var ve VIP destek hattini kullaniyor.",
     "context": "musteri profili"},
]
result = post(f"/v1/default/banks/{BANK_ID}/memories", {"items": more_facts})
print(json.dumps(result, indent=2, ensure_ascii=False))

show_memories(fact_type="world")

In [ ]:
consolidate_result = post(f"/v1/default/banks/{BANK_ID}/consolidate")
print("Consolidate operation:", consolidate_result)


def observations_ready():
    data = get(f"/v1/default/banks/{BANK_ID}/memories/list")
    obs = [m for m in data.get("items", []) if m.get("fact_type") == "observation"]
    return obs or None


wait_until(observations_ready, timeout=90, label="observation")
show_memories(fact_type="observation")

Check the **Observations** tab in the Admin UI. You should see one or more sentences that
were *not* in any single `retain` call — Hindsight wrote them by looking at several World
Facts together. Click one open: it lists its "Source Memories," i.e. exactly which World
Facts it was built from.

## 3. Mental Models — a standing question, kept answered

This is different from the other two. A Mental Model is **not** produced automatically.
You define it once, as a named question ("source_query"), and Hindsight keeps a
generated answer for it, refreshing on demand (or automatically after consolidation, if
you ask it to).

Think of it as a live dashboard tile: "Customer Profile" → Hindsight keeps that answer
up to date as new facts arrive, instead of you re-running a search every time.

In [ ]:
mm = post(f"/v1/default/banks/{BANK_ID}/mental-models", {
    "id": "musteri-profili",
    "name": "Musteri Profili",
    "source_query": "Ahmet hakkinda bildigimiz her seyi ozetle: tercihleri, gecmis sorunlari, musteri statusu.",
    "trigger": {"refresh_after_consolidation": True},
})
print(mm)


def mental_model_ready():
    data = get(f"/v1/default/banks/{BANK_ID}/mental-models/musteri-profili")
    content = data.get("content", "")
    return data if content and content != "Generating content..." else None


result = wait_until(mental_model_ready, timeout=90, label="mental model")
if result:
    print(result["content"])
else:
    # Timed out — show whatever state it's in anyway.
    print(get(f"/v1/default/banks/{BANK_ID}/mental-models/musteri-profili")["content"])

**Honest note:** in our own testing, this sometimes comes back saying it found no
relevant information, even though the World Facts above clearly exist. If that happens
here too, it is a real, reproducible behavior we ran into — not a mistake in this
notebook. Worth a closer look at Hindsight's own docs/issue tracker before relying on
Mental Models for a live demo.

## 4. Experience — an open question

The Admin UI describes **Experience** as *"the bank's own actions, interactions, and
first-person experiences."* We tried several ways to populate it and none worked, in
either this notebook's approach or the main chat app:

1. A `retain` call phrased as the agent's own first-person action.
2. A `retain` call shaped like a two-sided conversation transcript, grouped under one
   `document_id`.
3. Uploading that same transcript through the Admin UI's **+ Add Document** flow.

All three landed as **World Facts**, not **Experience**. The cell below repeats attempt
1 here so you can see the (non-)result directly; feel free to try your own variations.

In [ ]:
post(f"/v1/default/banks/{BANK_ID}/memories", {
    "items": [
        {
            "content": "Bugun Ahmet ile gorustum, kargo sikayetini dinledim ve VIP destek hattina yonlendirdim.",
            "context": "temsilci notu",
        }
    ]
})

print("Experience:")
show_memories(fact_type="experience")

print()
print("Nereye gitti? (World Facts'e bakalim)")
show_memories(fact_type="world")

If you figure out what actually triggers **Experience**, update this cell and this
markdown note — that is genuinely useful for the team.

## Summary

| Tab | How it's created | Reliable in our testing? |
|---|---|---|
| World Facts | Directly from `retain` | Yes |
| Observations | Auto-synthesized from multiple World Facts after `consolidate` | Yes |
| Mental Models | Explicitly defined (`source_query`), then `refresh`ed | Triggers, but sometimes returns "no data found" even with relevant facts present |
| Experience | Unknown — tried 3 methods, none worked | Open question |

Everything above happened without an LLM ever *deciding* to call a tool — we called the
REST API directly, so it is 100% repeatable for a live walkthrough. Compare this with the
main chat app, where an LLM decides when to call `retain`/`recall`/`reflect` — that is a
different (and much less predictable) part of the story.